# 🔬 Synthetic Data Generation (Local Jupyter + Lightning.ai)

Generate therapeutic AI training data via **Lightning.ai** inference credits.

**Why Lightning.ai:**
- Managed inference with 20+ models (GPT-5, Gemini, Llama, Nemotron)
- Pay-as-you-go / credit-based billing
- OpenAI-compatible `/v1/chat/completions` endpoint

**Prerequisites:** repo cloned, Python 3.11+, `LIGHTNING_API_KEY` env var or prompt below.

**Output:** JSONL files in `ai/training/data/generated/notebook_batch/`.

In [ ]:
# === Cell 1: Setup ===
import os, sys, json, subprocess
from pathlib import Path
from IPython.display import display, HTML

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from training.sdg_pipeline import build_parser, _build_nemo_config, _check_nemo_health, FAILED_CALL_ABORT_THRESHOLD
print(f'Repo: {REPO_ROOT}')
print(f'Abort threshold: {FAILED_CALL_ABORT_THRESHOLD}')
print('Ready.')

In [ ]:
# === Cell 2: Lightning.ai API Key ===
import getpass

api_key = os.environ.get('LIGHTNING_API_KEY', '')
if not api_key:
    api_key = getpass.getpass('Lightning.ai API Key: ')
os.environ['LIGHTNING_API_KEY'] = api_key

# Lightning.ai uses an OpenAI-compatible endpoint
LIT_ENDPOINT = 'https://lightning.ai/api/v1'
os.environ['NVIDIA_BASE_URL'] = LIT_ENDPOINT   # sdg_pipeline reads this as fallback
print(f'Key: ...{api_key[-8:]}')
print(f'Endpoint: {LIT_ENDPOINT}')

In [ ]:
# === Cell 3: Config ===
OUTPUT_DIR = REPO_ROOT / 'ai/training/data/generated/notebook_batch'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Lightning.ai model catalog includes:
#   openai/gpt-5, openai/gpt-5-mini, google/gemini-2.5-pro,
#   nvidia/llama-3.3-nemotron-super-49b-v1.5, lightning-ai/gpt-oss-120b, etc.
# Default: gpt-5-mini is cost-efficient for bulk generation.
LIT_MODEL    = 'openai/gpt-5-mini'   # swap to any model in your Lightning catalog
LIT_TIMEOUT  = 60                    # higher timeout for larger models
LIT_INTERVAL = 5.0                  # Lightning rate limits: 5s is usually safe
TARGET_COUNT  = 1000
MAX_ITER      = 5000
CLIP_VALIDITY = 0.0

CATEGORIES = [
    'somatic_therapy', 'attachment_disorders', 'personality_disorders',
    'dissociation', 'complicated_grief', 'eating_disorders',
    'ocd_intrusive_thoughts', 'narcissistic_abuse_recovery',
    'neurodivergent_mental_health', 'cultural_religious_contexts'
]
GENERATE_DPO = True
GENERATE_NF  = True

print(f'Output: {OUTPUT_DIR}')
print(f'Model: {LIT_MODEL}')
print(f'Categories: {len(CATEGORIES)} | DPO: {GENERATE_DPO} | NF: {GENERATE_NF}')

In [ ]:
# === Cell 4: Health Check ===
parser = build_parser()
args = parser.parse_args([
    '--scenario', 'dpo_preference_pairs', '--target_count', '1',
    '--max_iterations', '1', '--nemo_endpoint', LIT_ENDPOINT,
    '--nemo_api_key', api_key, '--nemo_model', LIT_MODEL,
    '--nemo_timeout', str(LIT_TIMEOUT), '--nemo_min_call_interval', str(LIT_INTERVAL),
    '--output_path', str(OUTPUT_DIR / '_health.jsonl')])
config = _build_nemo_config(args)
healthy = _check_nemo_health(config)
display(HTML(f"<span style='color:{'green' if healthy else 'red'}'>✅ Lightning.ai API Ready | ❌ API health check FAILED</span>"))

In [ ]:
# === Cell 5: Niche Categories ===
from tqdm.notebook import tqdm

pipeline = str(REPO_ROOT / 'ai/training/sdg_pipeline.py')

def _run(category, scenario='niche_category', extra_args=None, target=TARGET_COUNT, max_iter=MAX_ITER):
    out = OUTPUT_DIR / f'{category}.jsonl'
    existing = sum(1 for _ in open(out)) if out.exists() else 0
    need = max(0, target - existing)
    print(f'\n→ {category}: {existing}/{target}, need {need}')
    if need <= 0:
        return {'category': category, 'added': 0, 'total': existing}
    cmd = [
        sys.executable, pipeline, '--scenario', scenario,
        '--target_count', str(need), '--max_iterations', str(max_iter),
        '--nemo_endpoint', LIT_ENDPOINT,
        '--nemo_api_key', api_key, '--nemo_model', LIT_MODEL,
        '--nemo_timeout', str(LIT_TIMEOUT), '--nemo_min_call_interval', str(LIT_INTERVAL),
        '--min_clinical_validity', str(CLIP_VALIDITY), '--output_path', str(out)]
    if extra_args:
        cmd += extra_args
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ})
    log = OUTPUT_DIR / f'log_{category}.txt'
    log.write_text(r.stdout + '\n' + r.stderr)
    final = sum(1 for _ in open(out)) if out.exists() else 0
    print(f"  status: {'done' if r.returncode == 0 else 'FAILED'} | +{final-existing} | total {final}")
    return {'category': category, 'added': final-existing, 'total': final, 'log': str(log)}

results = []
for cat in tqdm(CATEGORIES, desc='Generating'):
    results.append(_run(cat, extra_args=['--category', cat]))

print('\n=== Niche Summary ===')
for r in results:
    print(f"{r['category']:35s} +{r['added']:4d} → {r['total']:5d}")

In [ ]:
# === Cell 6: DPO Preference Pairs ===
if GENERATE_DPO:
    r = _run('dpo_pairs', scenario='dpo_preference_pairs', target=TARGET_COUNT*2)
else:
    print('DPO generation disabled.')

In [ ]:
# === Cell 7: Nightmare Fuel ===
if GENERATE_NF:
    r = _run('nightmare_fuel', scenario='nightmare_fuel', target=TARGET_COUNT)
else:
    print('Nightmare fuel disabled.')

In [ ]:
# === Cell 8: Summary ===
from collections import Counter
total = 0
for f in OUTPUT_DIR.glob('*.jsonl'):
    if f.name.startswith('_'): continue
    n = sum(1 for _ in open(f))
    print(f'{f.name:40s} {n:6d}')
    total += n
print(f"{'Total':40s} {total:6d}")

sample_files = [f for f in OUTPUT_DIR.glob('*.jsonl') if not f.name.startswith('_')]
if sample_files:
    with open(sample_files[0]) as f:
        obj = json.loads(next(f))
    print(f'\n--- Peek: {sample_files[0].name} ---')
    print(json.dumps(obj, indent=2, ensure_ascii=False)[:1200])

In [ ]:
# === Cell 9: Merge to ChatML (optional) ===
merge_script = REPO_ROOT / 'ai/training/merge_final_dataset.py'
print(f'Merge script: {merge_script.exists()}')
print(f'\nTo merge all generated files into ChatML:')
print(f'  uv run python {merge_script}\n    --source_dirs {OUTPUT_DIR}\n    --output_dir {REPO_ROOT}/ai/training/notebook_merged')